# 03 - Enterprise Orchestration: Frontend Demo

## Overview

This tutorial provides a web UI to test the agents from Tutorials 01 and 02.

<p align="center">
  <img src="../../images/portal-policy-agent.png" alt="Policy Agent - RAG with metadata filtering" width="45%"/>
  <img src="../../images/portal-claims-agent.png" alt="Claims Agent - RLS with Row-Level Security" width="45%"/>
  <br/>
  <em>Same question, different answers: Each broker sees only their own data</em>
</p>


**Prerequisites**: Tutorials 01 and 02 must be completed (agents deployed, cleanup NOT run).

### What This Does

1. Reads agent ARNs, Cognito config, and Knowledge Base ID from Tutorials 01 and 02
2. Generates `backend/.env` with all required settings
3. Provides instructions to run Frontend + Backend

### Test Users

**Policy Agent (RAG - Tutorial 01)**
- john-regular  (Regular tier)
- sarah-platinum  (Platinum tier)

**Claims Agent (RLS - Tutorial 02)**
- mike-broker-a  (Broker A - sees CLM-A-* claims)
- emma-broker-b  (Broker B - sees CLM-B-* claims)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import boto3
import yaml
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")

## Step 1: Load Configuration from Tutorial 01 (RAG Agent)

In [ ]:
# Load Tutorial 01 config (RAG Agent)
rag_config_path = '../01-unstructured-multitenant-rag/.bedrock_agentcore.yaml'
if not os.path.exists(rag_config_path):
    raise Exception("Tutorial 01 not deployed. Run 01-unstructured-multitenant-rag first.")

with open(rag_config_path, 'r') as f:
    rag_config = yaml.safe_load(f)

rag_default = rag_config.get('default_agent')
rag_agent = rag_config['agents'][rag_default]
rag_agent_arn = rag_agent['bedrock_agentcore']['agent_arn']

# Get RAG Cognito config
rag_authorizer = rag_agent.get('authorizer_configuration', {}).get('customJWTAuthorizer', {})
rag_discovery_url = rag_authorizer.get('discoveryUrl', '')
rag_client_id = rag_authorizer.get('allowedClients', [None])[0]
rag_user_pool_id = rag_discovery_url.split('/')[-1] if rag_discovery_url else ''

# Get Knowledge Base ID from env_vars
rag_env_vars = rag_agent.get('env_vars', {})
knowledge_base_id = rag_env_vars.get('KNOWLEDGE_BASE_ID', '')

print(f"RAG Agent ARN: {rag_agent_arn}")
print(f"RAG Cognito User Pool: {rag_user_pool_id}")
print(f"RAG Cognito Client ID: {rag_client_id}")
print(f"Knowledge Base ID: {knowledge_base_id}")

## Step 2: Load Configuration from Tutorial 02 (RLS Agent)

In [ ]:
# Load Tutorial 02 config (RLS Agent)
rls_config_path = '../02-structured-multitenant-rls/.bedrock_agentcore.yaml'
if not os.path.exists(rls_config_path):
    raise Exception("Tutorial 02 not deployed. Run 02-structured-multitenant-rls first.")

with open(rls_config_path, 'r') as f:
    rls_config = yaml.safe_load(f)

rls_default = rls_config.get('default_agent')
rls_agent = rls_config['agents'][rls_default]
rls_agent_arn = rls_agent['bedrock_agentcore']['agent_arn']

# Get RLS Cognito config
rls_authorizer = rls_agent.get('authorizer_configuration', {}).get('customJWTAuthorizer', {})
rls_discovery_url = rls_authorizer.get('discoveryUrl', '')
rls_client_id = rls_authorizer.get('allowedClients', [None])[0]
rls_user_pool_id = rls_discovery_url.split('/')[-1] if rls_discovery_url else ''

print(f"RLS Agent ARN: {rls_agent_arn}")
print(f"RLS Cognito User Pool: {rls_user_pool_id}")
print(f"RLS Cognito Client ID: {rls_client_id}")

## Step 3: Generate backend/.env

In [ ]:
env_content = f"""# Auto-generated from Tutorial 03
AWS_REGION={region}

# Cognito - Tutorial 02 (RLS Agent)
RLS_COGNITO_USER_POOL_ID={rls_user_pool_id}
RLS_COGNITO_CLIENT_ID={rls_client_id}

# Cognito - Tutorial 01 (RAG Agent)
RAG_COGNITO_USER_POOL_ID={rag_user_pool_id}
RAG_COGNITO_CLIENT_ID={rag_client_id}

# Agent ARNs
RAG_AGENT_ARN={rag_agent_arn}
RLS_AGENT_ARN={rls_agent_arn}

# Knowledge Base (Tutorial 01)
RAG_KNOWLEDGE_BASE_ID={knowledge_base_id}
"""

with open('backend/.env', 'w') as f:
    f.write(env_content)

print("Created backend/.env")
print("="*60)
print(env_content)

## Step 4: Run Frontend and Backend

Open a terminal and run:

```bash
cd notebooks/03-enterprise-orchestration
./start.sh
```

Then open http://localhost:3000

### Test Users

**Policy Agent (RAG)**
- john-regular  - Regular tier policies
- sarah-platinum  - Platinum tier policies

**Claims Agent (RLS)**
- mike-broker-a  - Broker A claims
- emma-broker-b  - Broker B claims

In [ ]:
print("Configuration Summary")
print("="*60)
print(f"")
print(f"Policy Agent (RAG - Tutorial 01)")
print(f"  Agent: {rag_agent_arn.split('/')[-1]}")
print(f"  Cognito: {rag_user_pool_id}")
print(f"  KB: {knowledge_base_id}")
print(f"")
print(f"Claims Agent (RLS - Tutorial 02)")
print(f"  Agent: {rls_agent_arn.split('/')[-1]}")
print(f"  Cognito: {rls_user_pool_id}")
print("="*60)
print("")
print("Open a terminal and run:")
print("cd notebooks/03-enterprise-orchestration")
print("./start.sh")

# Congratulations!

You've successfully set up the enterprise orchestration demo with a React frontend and Flask backend!